In [ ]:
import io
from pathlib import Path
import re
import tempfile
import zipfile

import fiona
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
# folder with simplified (zip) shapes
shapes_folder = "./data_in/geo_sight/cod_shapes/"

# global geodatabases (zip): latest version of all countries, and historic
# versions (several per country)
gdb_folder = "./data_in/geo_sight/"
gdb_zips = {
    "latest": gdb_folder + "global_admin_boundaries_matched_latest.gdb.zip",
    "historic": gdb_folder + "global_admin_boundaries_matched_historic.gdb.zip",
}

# optional list of iso3 to restrict the run (None for all), e.g. ["AFG", "SOM"]
only_iso3 = None

# thresholds to flag geometries differing significantly
iou_threshold = 0.90  # min intersection over union
area_ratio_tol = 0.10  # max relative area difference (simplified vs reference)

# min overlap (fraction of the smaller unit) to link units across files
# NOTE: drops border slivers due to simplification
edge_min_frac = 0.05

# overlaps below this fraction (of both units) are boundary shifts between
# neighbours, not structural relations (split / merge / different subdivision)
boundary_shift_frac = 0.15

# equal area CRS to compute areas
equal_area_crs = "EPSG:6933"

In [ ]:
def read_simplified(zip_path):
    """GeoDataFrame from the geojson inside the simplified zip file."""
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        member = [
            n for n in zip_file.namelist()
            if n.lower().endswith(".geojson")
        ][0]
        return gpd.read_file(io.BytesIO(zip_file.read(member)))


def get_col(gdf, name):
    """Column in gdf matching name case insensitive (None if missing)."""
    found = [c for c in gdf.columns if c.lower() == name.lower()]
    return found[0] if found else None


def prepare(gdf, pcode_col):
    """Valid geometries dissolved by pcode, in equal area CRS."""
    gdf = gdf[[pcode_col, "geometry"]].copy()
    gdf[pcode_col] = gdf[pcode_col].astype(str).str.strip()
    gdf["geometry"] = gdf.make_valid()
    return gdf.dissolve(by=pcode_col).to_crs(equal_area_crs)


def overlap_table(gdf_s, gdf_o, pcode_s, pcode_o):
    """Overlaps between units of both files (equal area CRS).

    Keeps pairs where the intersection is at least edge_min_frac of the
    smaller unit. Returns edges and the pcodes of each file.
    """
    geo_s = prepare(gdf_s, pcode_s).geometry
    geo_o = prepare(gdf_o, pcode_o).geometry

    idx_s, idx_o = geo_o.sindex.query(geo_s.values, predicate="intersects")
    inter = geo_s.iloc[idx_s].intersection(
        geo_o.iloc[idx_o], align=False
    ).area.to_numpy()
    area_s = geo_s.area.to_numpy()[idx_s]
    area_o = geo_o.area.to_numpy()[idx_o]

    edges = pd.DataFrame({
        "pcode_s": geo_s.index[idx_s],
        "pcode_o": geo_o.index[idx_o],
        "iou": inter / (area_s + area_o - inter),
        "frac_s": inter / area_s,
        "frac_o": inter / area_o,
        "area_ratio": area_s / area_o,
    })
    edges = edges[
        edges[["frac_s", "frac_o"]].max(axis=1) >= edge_min_frac
    ].reset_index(drop=True)
    return edges, list(geo_s.index), list(geo_o.index)


def classify_relations(edges, pcodes_s, pcodes_o):
    """Crosswalk of units across files from connected overlaps.

    Overlaps below boundary_shift_frac (both fractions) are not used for
    relations; returned as a separate table of boundary shifts.

    relation: same_pcode, pcode_changed, pcode_interchanged (1 to 1);
    split (1 simplified to n original); merge (n to 1); complex (n to m);
    no_counterpart_s / no_counterpart_o (no overlap in the other file).
    """
    shift = edges[["frac_s", "frac_o"]].max(axis=1) < boundary_shift_frac
    shifts = edges[shift].reset_index(drop=True)
    edges = edges[~shift].reset_index(drop=True)

    node_s = {p: i for i, p in enumerate(pcodes_s)}
    node_o = {p: len(pcodes_s) + i for i, p in enumerate(pcodes_o)}
    n_nodes = len(pcodes_s) + len(pcodes_o)
    rows = edges["pcode_s"].map(node_s).to_numpy(dtype=int)
    cols = edges["pcode_o"].map(node_o).to_numpy(dtype=int)
    _, labels = connected_components(
        coo_matrix((np.ones(len(edges)), (rows, cols)), shape=(n_nodes, n_nodes)),
        directed=False,
    )

    cw = edges.assign(component=labels[rows])
    n_s = cw.groupby("component")["pcode_s"].transform("nunique")
    n_o = cw.groupby("component")["pcode_o"].transform("nunique")
    cw["relation"] = np.select(
        [(n_s == 1) & (n_o == 1), n_s == 1, n_o == 1],
        ["one_to_one", "split", "merge"],
        default="complex",
    )

    # one to one: same pcode, changed, or interchanged (pcodes reused)
    one = cw["relation"] == "one_to_one"
    changed = one & (cw["pcode_s"] != cw["pcode_o"])
    cw.loc[one, "relation"] = "same_pcode"
    cw.loc[changed, "relation"] = "pcode_changed"
    interchanged = (
        changed
        & cw["pcode_s"].isin(cw.loc[changed, "pcode_o"])
        & cw["pcode_o"].isin(cw.loc[changed, "pcode_s"])
    )
    cw.loc[interchanged, "relation"] = "pcode_interchanged"

    # units without overlap in the other file
    lone_s = sorted(set(pcodes_s) - set(cw["pcode_s"]))
    lone_o = sorted(set(pcodes_o) - set(cw["pcode_o"]))
    lone = pd.concat([
        pd.DataFrame({"pcode_s": lone_s, "relation": "no_counterpart_s"}),
        pd.DataFrame({"pcode_o": lone_o, "relation": "no_counterpart_o"}),
    ], ignore_index=True)
    lone["component"] = labels.max() + 1 + np.arange(len(lone))

    cw = pd.concat([cw, lone], ignore_index=True).sort_values(
        ["relation", "component"]
    ).reset_index(drop=True)
    return cw, shifts


one_to_one_rel = ["same_pcode", "pcode_changed", "pcode_interchanged"]


def print_reconciliation(cw, shifts, max_rows=20):
    """Proposal to reconcile pcodes across files from geometry matches."""
    if set(cw["relation"]) <= {"same_pcode"}:
        print("Pcodes consistent across files, nothing to reconcile")

    cases = [
        ("pcode_changed", "Same shape, different pcode: recode original "
         "pcode (pcode_o) to simplified pcode (pcode_s)",
         ["pcode_o", "pcode_s", "iou"]),
        ("pcode_interchanged", "Same shapes, pcodes interchanged between "
         "units: recode one file with this mapping",
         ["pcode_o", "pcode_s", "iou"]),
        ("split", "Split: a simplified unit is divided in the reference; "
         "aggregate pcode_o (reference) to pcode_s (frac_o ~ share of pcode_o in pcode_s)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("merge", "Merge: several simplified units are one in the reference; "
         "aggregate pcode_s to pcode_o (reference) (frac_s ~ share of pcode_s in pcode_o)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("complex", "Different subdivision (n to m): no clean crosswalk, "
         "manual review (area fractions as allocation weights)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("no_counterpart_s", "Units only in simplified (no overlap in reference)",
         ["pcode_s"]),
        ("no_counterpart_o", "Units only in reference (no overlap in simplified)",
         ["pcode_o"]),
        ("shifts", "Boundary shifts between neighbours (not structural, "
         "see IoU of the units involved)",
         ["pcode_s", "pcode_o", "frac_s", "frac_o"]),
    ]
    for relation, text, cols in cases:
        sub = shifts if relation == "shifts" else cw[cw["relation"] == relation]
        if sub.empty:
            continue
        print(f"{text} [{len(sub)} rows]")
        print(sub[cols].round(3).head(max_rows).to_string(index=False))
        if len(sub) > max_rows:
            print(f"... {len(sub) - max_rows} more rows")

In [ ]:
def read_gdb_layer(zip_path, level):
    """Admin level layer of a zipped geodatabase, in EPSG:4326."""
    with tempfile.TemporaryDirectory() as temp_dir:
        gdb_path = Path(temp_dir) / "data.gdb"
        gdb_path.mkdir()
        with zipfile.ZipFile(zip_path, "r") as zip_file:
            zip_file.extractall(gdb_path)

        # layer named after the admin level; else positional as in
        # read_gdb_global.py
        layers = fiona.listlayers(gdb_path)
        named = [
            l for l in layers
            if re.search(rf"adm(in)?_?{level}(?!\d)", l, re.IGNORECASE)
        ]
        layer = named[0] if named else layers[level - 1]
        print(f"Reading layer '{layer}' from {Path(zip_path).name}")
        return gpd.read_file(gdb_path, layer=layer).to_crs("EPSG:4326")


# reference geodatabase layers by admin level, read once and stored
ref_cache = {"latest": {}, "historic": {}}


def get_reference(source, level):
    """Admin level layer of the latest / historic geodatabase (cached)."""
    if level not in ref_cache[source]:
        ref_cache[source][level] = read_gdb_layer(gdb_zips[source], level)
    return ref_cache[source][level]


def version_number(version):
    """Number in version label, e.g. "v02" -> 2 (-1 if none)."""
    found = re.search(r"\d+", str(version))
    return int(found.group()) if found else -1


def reference_candidates(source, iso3, level):
    """Reference shapes of a country for the admin level, latest version first.

    List of (label, GeoDataFrame, pcode column); empty if the country has no
    shapes (or no pcode column) in the geodatabase layer.
    """
    gdf = get_reference(source, level)
    iso_col = get_col(gdf, "iso3")
    pcode_col = get_col(gdf, f"adm{level}_pcode")
    if iso_col is None or pcode_col is None:
        print(f"Note columns 'iso3' / 'adm{level}_pcode' not found in "
              f"{source} layer, columns: {list(gdf.columns)}")
        return []
    gdf = gdf[gdf[iso_col].astype(str).str.upper() == iso3.upper()]
    if gdf.empty:
        return []

    version_col = get_col(gdf, "version")
    if version_col is None:
        return [(source, gdf, pcode_col)]
    versions = sorted(gdf[version_col].dropna().unique(),
                      key=version_number, reverse=True)
    return [
        (f"{source} {v}", gdf[gdf[version_col] == v], pcode_col)
        for v in versions
    ]

In [ ]:
rel_order = [
    "same_pcode", "pcode_changed", "pcode_interchanged", "split",
    "merge", "complex", "no_counterpart_s", "no_counterpart_o",
]


def compare(gdf_s, gdf_r, pcode_s, pcode_r, verbose=True):
    """Compare simplified shapes against reference shapes (same country).

    Returns metrics, crosswalk (cw) and boundary shifts of the comparison.
    """
    set_s = set(gdf_s[pcode_s].astype(str).str.strip())
    set_r = set(gdf_r[pcode_r].astype(str).str.strip())
    pcodes_equal = set_s == set_r
    if verbose:
        print(f"Pcodes: {len(set_s)} simplified, {len(set_r)} reference")
        if pcodes_equal:
            print("Pcodes are the same in both files")
        else:
            only_s = sorted(set_s - set_r)
            only_r = sorted(set_r - set_s)
            print(f"Pcodes only in simplified ({len(only_s)}): {only_s[:20]}")
            print(f"Pcodes only in reference ({len(only_r)}): {only_r[:20]}")

    # match shapes across files (regardless of pcodes)
    edges, all_s, all_r = overlap_table(gdf_s, gdf_r, pcode_s, pcode_r)
    cw, shifts = classify_relations(edges, all_s, all_r)
    rel_counts = cw.groupby("relation")["component"].nunique()

    # shape similarity for 1 to 1 matches (same shape, regardless of pcode)
    res = cw[cw["relation"].isin(one_to_one_rel)]
    flagged = res[
        (res["iou"] < iou_threshold)
        | ((res["area_ratio"] - 1).abs() > area_ratio_tol)
    ]
    if verbose:
        print(f"Boundary shifts (minor overlaps between units): {len(shifts)}")
        print("Matches by relation (n components):")
        print(rel_counts.to_string())
        if len(res) > 0:
            print(
                f"1 to 1 IoU: median {res['iou'].median():.4f}, "
                f"min {res['iou'].min():.4f}; "
                f"area ratio: min {res['area_ratio'].min():.4f}, "
                f"max {res['area_ratio'].max():.4f}"
            )
            print(f"Flagged (IoU < {iou_threshold} or area diff > "
                  f"{area_ratio_tol:.0%}): {len(flagged)}")
            if len(flagged) > 0:
                print(flagged.sort_values("iou")[
                    ["pcode_s", "pcode_o", "iou", "area_ratio"]
                ].head(5).round(4))

    return {
        "n_simplified": len(set_s),
        "n_reference": len(set_r),
        "pcodes_equal": pcodes_equal,
        **{r: rel_counts.get(r, 0) for r in rel_order},
        "n_boundary_shift": len(shifts),
        "median_iou": res["iou"].median() if len(res) else np.nan,
        "min_iou": res["iou"].min() if len(res) else np.nan,
        "n_flagged": len(flagged),
        "one_to_one": is_one_to_one(cw),
        "cw": cw,
        "shifts": shifts,
    }


def is_one_to_one(cw):
    """Same shapes and same pcodes: every unit matches one to one, same pcode."""
    return len(cw) > 0 and bool((cw["relation"] == "same_pcode").all())


def closeness(result):
    """Sort key of an attempt: more same pcode matches, fewer other units."""
    others = sum(result[r] for r in rel_order[1:])
    return (result["same_pcode"], -others)

In [ ]:
def parse_name(name):
    """(iso3, adm level) from a simplified zip name (None if not parsed)."""
    parsed = re.match(r"geojson_([A-Za-z]{3})_.*?adm(\d+)_simpl", name)
    return parsed.groups() if parsed else None


zip_paths = []
for zip_path in sorted(Path(shapes_folder).glob("geojson_*_adm*_simpl*.zip")):
    parsed = parse_name(zip_path.name)
    if parsed is None:
        print(f"Note file name not parsed (iso3 / adm level), skipping: "
              f"{zip_path.name}")
    elif only_iso3 is None or parsed[0].upper() in only_iso3:
        zip_paths.append((zip_path, parsed[0].upper(), int(parsed[1])))

# admin levels needed from the reference: read only those, once
levels_needed = sorted({level for _, _, level in zip_paths})
print(f"{len(zip_paths)} simplified files, admin levels: {levels_needed}")
for level in levels_needed:
    get_reference("latest", level)

In [ ]:
summary_rows = []
attempts = {}  # per file: result of each reference version tried
crosswalks = {}  # crosswalk per (file, reference), to inspect / export
drop_keys = ("cw", "shifts", "gdf_r", "pcode_r", "one_to_one")

for zip_path, iso3, level in zip_paths:
    print("=" * 70)
    print(zip_path.name)

    gdf_s = read_simplified(zip_path)
    pcode_s = get_col(gdf_s, f"adm{level}_pcode")
    if pcode_s is None:
        print(f"Note column 'adm{level}_pcode' not found (case insensitive) "
              f"in simplified, columns: {list(gdf_s.columns)}")
        continue

    # latest geodatabase first, then historic versions (latest to oldest),
    # only until the first one to one match
    tried = []
    matched = None
    for source in ["latest", "historic"]:
        for label, gdf_r, pcode_r in reference_candidates(source, iso3, level):
            result = compare(gdf_s, gdf_r, pcode_s, pcode_r, verbose=False)
            result.update(reference=label, gdf_r=gdf_r, pcode_r=pcode_r)
            tried.append(result)
            crosswalks[(zip_path.name, label)] = result["cw"]
            print(f"Reference {label}: {result['n_reference']} units, "
                  f"one to one match: {result['one_to_one']}")
            if result["one_to_one"]:
                matched = result
                break
        if matched is not None:
            break
    if not tried:
        print(f"Note no shapes for {iso3} adm{level} in the geodatabases")
        continue
    attempts[zip_path.name] = tried

    if matched is not None:
        # details of the matched reference
        print(f"Matched reference: {matched['reference']}")
        compare(gdf_s, matched["gdf_r"], pcode_s, matched["pcode_r"])
        print_reconciliation(matched["cw"], matched["shifts"])
        selected = matched
    else:
        # mismatch details in the summary at the end of the run
        selected = max(tried, key=closeness)
        print("No one to one match in any reference, closest: "
              f"{selected['reference']}")

    summary_rows.append({
        "simplified": zip_path.name,
        "reference": selected["reference"] if matched else "none",
        "closest": selected["reference"],
        **{k: v for k, v in selected.items()
           if k not in drop_keys and k != "reference"},
    })

In [ ]:
# summary of all analysed files
# NOTE: crosswalks[(<zip name>, <reference>)] has the unit matches per file
summary = pd.DataFrame(summary_rows)
summary.round(4)

In [ ]:
# files not matching the latest geodatabase: every reference tried
# (latest to oldest version) and the one finally matched, if any
rows = []
for name, tried in attempts.items():
    if tried[0]["one_to_one"] and tried[0]["reference"].startswith("latest"):
        continue
    for r in tried:
        rows.append({
            "simplified": name,
            **{k: v for k, v in r.items() if k not in drop_keys},
            "matched": r["one_to_one"],
        })
not_latest = pd.DataFrame(rows)
not_latest.round(4)

In [ ]:
# files without a one to one match in any reference: mismatches by relation
# (pcode changes, splits, merges, complex, units without counterpart, shifts)
# against each reference tried, latest to oldest
for name, tried in attempts.items():
    if any(r["one_to_one"] for r in tried):
        continue
    print("=" * 70)
    print(name)
    for r in tried:
        print("-" * 70)
        print(f"Reference: {r['reference']} ({r['n_reference']} units, "
              f"{r['n_simplified']} simplified)")
        counts = {k: r[k] for k in rel_order if r[k]}
        print(f"Matches by relation (n components): {counts}; "
              f"boundary shifts: {r['n_boundary_shift']}")
        print(f"1 to 1 IoU: median {r['median_iou']:.4f}, "
              f"min {r['min_iou']:.4f}; flagged: {r['n_flagged']}")
        print_reconciliation(r["cw"], r["shifts"], max_rows=10)
    best = max(tried, key=closeness)
    print(f"Closest reference: {best['reference']} "
          f"({best['same_pcode']} same pcode matches of "
          f"{best['n_simplified']} simplified units)")